# Ideological transitions and shocks across a country network

Notebook for running the supplied project. Download the repository as a ZIP from GitHub (for example, `Teuken-Markov.zip`). In Colab, run the upload cell and select the ZIP; GitHub archives with a top-level folder such as `Teuken-Markov-main` are supported. In local Jupyter, extract the archive and open this notebook from the project folder containing `run.py`. Existing project archives are also accepted, regardless of their filename.

The bundled reference run uses DPI 1975–2023 and 31 countries. Its evaluation contains 604 cases and 60 changes; scenarios begin in January 2023. These are conditional scenarios, not forecasts of current governments. Read README and SOURCES before interpreting the results.

In [ ]:
from pathlib import Path
import io, zipfile

candidates = [Path.cwd(), Path.cwd()/"Teuken-Markov", Path.cwd()/"americas_markov",
              Path("/content/Teuken-Markov"), Path("/content/americas_markov")]
# Also recognize extracted GitHub archives with a branch name in the folder name.
candidates.extend(path.parent for path in sorted(Path.cwd().glob("*/run.py")))
root = next((p for p in candidates if (p/"run.py").is_file()), None)
if root is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Extract the project ZIP and open this notebook from the folder containing run.py."
        ) from exc
    uploaded = files.upload()
    selected = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if selected is None:
        raise ValueError("Select the project ZIP, for example Teuken-Markov.zip.")
    destination = Path("/content").resolve()
    with zipfile.ZipFile(io.BytesIO(uploaded[selected])) as archive:
        for item in archive.infolist():
            target = (destination/item.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError("Unsafe path inside the ZIP archive.")
        archive.extractall(destination)
        # Locate the project without assuming a fixed archive or top-level folder name.
        project_folders = sorted({
            (destination/Path(item.filename).parent).resolve()
            for item in archive.infolist()
            if Path(item.filename).name == "run.py"
            and (destination/Path(item.filename).parent/"requirements.txt").is_file()
        })
    if len(project_folders) != 1:
        raise ValueError("The ZIP must contain exactly one project folder with run.py and requirements.txt.")
    root = project_folders[0]
if not (root/"run.py").is_file():
    raise FileNotFoundError("The project folder is missing run.py.")
print("Project folder:", root)


## Install dependencies

`requirements.txt` allows compatible versions. `requirements-lock.txt` records the versions used with Python 3.12; Python 3.11 or later is required.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(root/"requirements.txt")], check=True)


## Verify or refresh the sources

The data used for the reference run are already included. This cell uses the cache and retrieves the original COW files if they were omitted from the ZIP. Add `--refresh` to download newer versions; the numerical results may change. You can skip this cell and run the analysis offline.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--download"], check=True)


## Run the six models and scenarios

Imputation and scaling are fitted exclusively within each training set. Validation advances chronologically by year. The simulation clock is annual and does not use future election dates. Unknown states are not recoded as center.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--offline", "--run", "--simulate", "--runs", "500", "--horizon", "6", "--seed", "2026"], check=True)


## Compare results

Lower Brier scores and log loss are better. The risk of an ideological change is `1 - probability of the initial state`; AUC and average precision evaluate that risk without selecting a threshold using future outcome labels. Accuracy near 90% in the bundled reference run primarily reflects persistence: the maximum-probability class predictions detected none of the 60 changes.

In [ ]:
import pandas as pd
metrics = pd.read_csv(root/"outputs/main/metrics.csv")
display(metrics[metrics["subset"]=="all"][["model","n","brier_multiclass","log_loss_clipped","brier_change","roc_auc_change","average_precision_change","accuracy"]])
print((root/"outputs/main/RESULTS.txt").read_text())


In [ ]:
from IPython.display import Image, display
for name in ["model_comparison", "calibration", "scenarios"]:
    display(Image(filename=str(root/"outputs/main"/(name+".png"))))


## Optional sensitivity analyses

The supplied project includes an analysis with land and maritime connections and a run using land borders only. To re-estimate the latter, use the next cell. `--competitive-only` restricts origin observations to electoral competitiveness EIEC >= 6; this is a different sample and must be compared with its own baseline.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--offline", "--run", "--land-only"], check=True)


## Modify assumptions and save

Edit the `SCENARIOS` dictionary in `src/americas_markov/simulation.py`: the recession scenario sets global growth to −2%, while the energy and food scenario sets oil price growth to +50% and food price growth to +20%, for two steps. The baseline keeps prices constant and the COVID indicator at zero. The code does not estimate an effect of the war in Ukraine. Changing these assumptions produces new conditional scenarios; it does not establish causality.

To extend the study, update the political source and construct a calendar of opportunities for change that was known at each forecast date. The annual series contains few changes and substantial institutional persistence.

In [ ]:
import shutil
result_archive = shutil.make_archive(str(root.parent/"Teuken-Markov_run"), "zip", root_dir=root.parent, base_dir=root.name)
print("Results archive:", result_archive)
try:
    from google.colab import files
    files.download(result_archive)
except ImportError:
    pass
